In [2]:
# dect_slr_extract.py

import os
import json
from pathlib import Path

import fitz  # PyMuPDF
import pandas as pd
from openai import OpenAI

from reportlab.lib.pagesizes import A4
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, PageBreak, Table, TableStyle
from reportlab.lib.styles import getSampleStyleSheet
from reportlab.lib import colors


PDF_FOLDER = Path.cwd()
OUTPUT_EXCEL = "DECT_SLR_Extraction.xlsx"
OUTPUT_PDF = "DECT_SLR_Extraction_Report.pdf"

client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))


def extract_pdf_text(pdf_path, max_pages=12):
    text_parts = []
    try:
        doc = fitz.open(pdf_path)
        for page_num in range(min(len(doc), max_pages)):
            text_parts.append(doc[page_num].get_text("text"))
        doc.close()
    except Exception as e:
        return f"ERROR reading PDF: {e}"

    text = "\n".join(text_parts)
    text = " ".join(text.split())
    return text[:50000]


def extract_slr_fields(file_name, text):
    prompt = f"""
You are extracting data for a Systematic Literature Review on DECT-2020 NR / DECT NR+.

Return ONLY valid JSON. Do not use markdown.

Fields:
{{
  "title": "",
  "authors": "",
  "year": "",
  "publication_type": "",
  "venue": "",
  "include_in_slr": "Yes/No/Maybe",
  "exclusion_reason": "",
  "research_category": "",
  "technical_focus": "",
  "research_objective": "",
  "methodology": "",
  "evaluation_type": "",
  "hardware_platform": "",
  "software_tools": "",
  "frequency_band": "",
  "network_topology": "",
  "performance_metrics": "",
  "main_results": "",
  "key_contribution": "",
  "limitations": "",
  "future_work": "",
  "research_gap": "",
  "quality_objective_0_2": "",
  "quality_method_0_2": "",
  "quality_contribution_0_2": "",
  "quality_validation_0_2": "",
  "quality_limitations_0_2": "",
  "quality_total_0_10": ""
}}

Inclusion rule:
Include only if the paper primarily focuses on DECT-2020 NR / DECT NR+.
Exclude if it only mentions DECT-2020 NR briefly.

File name:
{file_name}

Paper text:
{text}
"""

    try:
        response = client.chat.completions.create(
            model="gpt-4.1-mini",
            messages=[
                {"role": "system", "content": "You extract structured SLR data from academic papers."},
                {"role": "user", "content": prompt},
            ],
            temperature=0,
        )

        content = response.choices[0].message.content.strip()

        if content.startswith("```"):
            content = content.replace("```json", "").replace("```", "").strip()

        return json.loads(content)

    except Exception as e:
        return {
            "title": file_name,
            "include_in_slr": "Maybe",
            "exclusion_reason": f"Extraction failed: {e}",
        }


def create_pdf_report(records, output_pdf):
    doc = SimpleDocTemplate(output_pdf, pagesize=A4)
    styles = getSampleStyleSheet()
    story = []

    story.append(Paragraph("DECT-2020 NR SLR Extraction Report", styles["Title"]))
    story.append(Spacer(1, 12))

    for i, rec in enumerate(records, start=1):
        story.append(Paragraph(f"{i}. {rec.get('title', 'Unknown Title')}", styles["Heading2"]))

        table_data = [
            ["Include", rec.get("include_in_slr", "")],
            ["Category", rec.get("research_category", "")],
            ["Methodology", rec.get("methodology", "")],
            ["Evaluation", rec.get("evaluation_type", "")],
            ["Platform", rec.get("hardware_platform", "")],
            ["Metrics", rec.get("performance_metrics", "")],
            ["Contribution", rec.get("key_contribution", "")],
            ["Limitations", rec.get("limitations", "")],
            ["Research Gap", rec.get("research_gap", "")],
            ["Quality Score", str(rec.get("quality_total_0_10", ""))],
        ]

        table = Table(table_data, colWidths=[100, 380])
        table.setStyle(TableStyle([
            ("BACKGROUND", (0, 0), (0, -1), colors.lightgrey),
            ("GRID", (0, 0), (-1, -1), 0.25, colors.grey),
            ("VALIGN", (0, 0), (-1, -1), "TOP"),
        ]))

        story.append(table)
        story.append(Spacer(1, 16))

        if i % 3 == 0:
            story.append(PageBreak())

    doc.build(story)


def main():
    if not os.environ.get("OPENAI_API_KEY"):
        raise RuntimeError("OPENAI_API_KEY is not set.")

    pdf_files = sorted(PDF_FOLDER.glob("*.pdf"))

    if not pdf_files:
        print(f"No PDF files found in: {PDF_FOLDER}")
        return

    print(f"Found {len(pdf_files)} PDFs in {PDF_FOLDER}")

    records = []

    for pdf in pdf_files:
        print(f"Processing: {pdf.name}")
        text = extract_pdf_text(pdf)
        result = extract_slr_fields(pdf.name, text)
        result["file_name"] = pdf.name
        records.append(result)

        pd.DataFrame(records).to_excel(OUTPUT_EXCEL, index=False)

    create_pdf_report(records, OUTPUT_PDF)

    print("Done.")
    print(f"Excel saved to: {OUTPUT_EXCEL}")
    print(f"PDF report saved to: {OUTPUT_PDF}")


if __name__ == "__main__":
    main()

OpenAIError: Missing credentials. Please pass an `api_key`, `workload_identity`, `admin_api_key`, or set the `OPENAI_API_KEY` or `OPENAI_ADMIN_KEY` environment variable.

In [4]:
import os

os.environ["OPENAI_API_KEY"] = "sk-proj-YOUR_REAL_KEY_HERE"

print(os.environ["OPENAI_API_KEY"][:15])

sk-proj-YOUR_RE


In [8]:
from pathlib import Path

folder = Path("/home/ashuqullah-alizai/Documents/DECT_Papers/DECT_SLR/DECT_LitratureReviews/files/onepage_summaries")

output_file = folder / "combined.txt"

txt_files = sorted(folder.glob("*.txt"))

with open(output_file, "w", encoding="utf-8") as outfile:
    for txt_file in txt_files:
        if txt_file.name == output_file.name:
            continue

        outfile.write("=" * 100 + "\n")
        outfile.write(f"FILE: {txt_file.name}\n")
        outfile.write("=" * 100 + "\n\n")

        with open(txt_file, "r", encoding="utf-8", errors="ignore") as infile:
            outfile.write(infile.read())

        outfile.write("\n\n\n")

print(f"Combined {len(txt_files)} files into:\n{output_file}")

Combined 58 files into:
/home/ashuqullah-alizai/Documents/DECT_Papers/DECT_SLR/DECT_LitratureReviews/files/onepage_summaries/combined.txt
